## Preparacion para Lectura, RAG y Agente

### 1. Instalación base
Estas celdas instalan las dependencias principales del notebook: LangChain, OpenAI, loaders de PDF y utilidades de text splitting.

In [ ]:
pip install --upgrade pip

In [ ]:
%pip install -q langchain langchain-openai langchain-community pypdf langchain-text-splitters

In [ ]:
%pip install -q -U langchain-core

# Instalamos NLTK (Natural Language Toolkit), una librería de NLP en Python.

En tu notebook se usa en la celda siguiente para bajar recursos de tokenización y tagging:

### 2. Dependencias para embeddings y experimentación
Este bloque instala PyTorch, Transformers y extensiones experimentales de LangChain que vamos a usar más adelante para pruebas y comparación de estrategias.

In [ ]:
%pip install -q nltk

In [ ]:
%pip install -q "torch==2.2.2" "transformers<5"

In [ ]:
%pip install -q -U langchain-experimental

### 3. Verificación del entorno
Estas celdas validan que las versiones instaladas realmente estén disponibles en el kernel activo y descargan recursos auxiliares de NLTK.

In [ ]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

In [ ]:

import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

#Preparamos para importar variables de entorno

### 4. Variables de entorno y credenciales
Este bloque carga la configuración local y, si falta la API key, la pide de forma interactiva para no dejarla hardcodeada en el notebook.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
import getpass

if not openai_api_key:
    openai_api_key = getpass.getpass()

### 5. Carga del documento fuente
Acá se abre el PDF de prueba, se extrae el texto y se hace una primera validación para saber si el archivo trae texto embebido o si más adelante conviene OCR.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_para_ocr_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf"
pdf_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104046.pdf"

docs = PyPDFLoader(pdf_para_ocr_path).load()
pdf_text = "\n\n".join(doc.page_content for doc in docs).strip()

print(f"Paginas cargadas: {len(docs)}")
print(f"Chars extraidos: {len(pdf_text)}")

if not pdf_text:
    print("Este PDF parece escaneado o sin texto embebido. Usa OCR con el endpoint de ingesta y pdf_para_ocr_path.")
    

In [ ]:
pdf_text

In [ ]:
%pip install -q "torch==2.2.2" "transformers<5"

## Modelo de Embeddings y Tokenizador
- Vamos a Usar Transformers

### 6. Modelo local de embeddings para laboratorio
Estas celdas cargan un modelo local solo para experimentar y comparar. No es la ruta principal de producción si seguimos con OpenAI.

In [ ]:
import torch
AutoTokenizer = transformers.AutoTokenizer
AutoModel = transformers.AutoModel

print("torch:", torch.__version__)

emb_tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
emb_model = AutoModel.from_pretrained("intfloat/multilingual-e5-small")
print("Modelo cargado OK")

### 7. Splitter semántico con OpenAI
Acá se crea el splitter semántico basado en embeddings de OpenAI. Todavía no define el chunk final de producción; por ahora nos sirve para experimentar después de limpiar el texto legal.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker  # type: ignore[reportMissingImports]
from langchain_openai.embeddings import OpenAIEmbeddings

semantic_openai_splitter = SemanticChunker(OpenAIEmbeddings(api_key=openai_api_key))

### Módulo A — Leyes Sancionadas (Infoleg)
**Fuente de Verdad Firme.** Si el documento es una sanción, se carga directamente desde Infoleg (`texact.htm`). El texto ya es limpio y estructurado: **no se corre OCR, no se limpia nada**.

Flujo:
- `tipo_documento = "sancion"` → carga desde Infoleg, saltea la Sección 8
- `tipo_documento = "proyecto"` → continúa con el pipeline OCR de la Sección 8

In [ ]:
%pip install -q beautifulsoup4 requests

In [7]:
import re
import hashlib
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# ─────────────────────────────────────────────────────────────────────────────
# URLs base de Infoleg
# ─────────────────────────────────────────────────────────────────────────────
_INFOLEG_BASE = "https://servicios.infoleg.gob.ar/infolegInternet"
_INFOLEG_ATTACH_BASE = f"{_INFOLEG_BASE}/anexos"


def _buscar_id_norma_infoleg(numero_ley: int, tipo_norma: int = 1, anio_sancion: str = "") -> tuple[int, str]:
    """
    Paso 1: buscar norma por tipo+numero en buscarNormas.do y extraer el id
    del primer <a href="/infolegInternet/verNorma.do?id=XXXXX"> del HTML.
    tipo_norma=1 corresponde a LEY.
    """
    search_url = (
        f"{_INFOLEG_BASE}/buscarNormas.do?tipoNorma={tipo_norma}"
        f"&numero={numero_ley}&anioSancion={anio_sancion}"
    )

    resp = requests.get(search_url, timeout=20)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding or "utf-8"

    # Extrae el id del primer <a href="/infolegInternet/verNorma.do?id=..."> del HTML
    m = re.search(
        r'<a href="/infolegInternet/verNorma\.do(?:;[^?]*)?\?id=(\d+)"',
        resp.text,
        re.IGNORECASE,
    )
    if not m:
        raise ValueError(
            f"No se pudo extraer idNorma para la ley {numero_ley} desde buscarNormas.do"
        )

    return int(m.group(1)), search_url


def _url_ver_norma(infoleg_id: int) -> str:
    return f"{_INFOLEG_BASE}/verNorma.do?id={infoleg_id}"


def _url_texact_fallback(infoleg_id: int) -> str:
    """Fallback clásico por rango de anexos cuando no aparece link directo en verNorma."""
    low = (infoleg_id // 10000) * 10000
    high = low + 9999
    return f"{_INFOLEG_ATTACH_BASE}/{low}-{high}/{infoleg_id}/texact.htm"


def _extraer_url_texact_desde_ver_norma(ver_norma_html: str, ver_norma_url: str) -> str | None:
    """Intenta encontrar un href a texact.htm en la página de verNorma."""
    soup = BeautifulSoup(ver_norma_html, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "texact.htm" in href.lower():
            return urljoin(ver_norma_url, href)

    return None


def _parse_infoleg_html(
    soup: BeautifulSoup,
    numero_ley: int,
    infoleg_id: int,
    source_url: str,
    sha256_hash: str,
) -> list[dict]:
    """Segmenta el HTML de Infoleg en unidades por artículo."""
    article_re = re.compile(r"^\s*(art[íi]culo|art\.)\s*[\dA-Za-z°º]+", re.IGNORECASE)
    heading_re = re.compile(r"^\s*(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", re.IGNORECASE)

    base_meta = {
        "source": "INFOLEG",
        "status": "VIGENTE",
        "law_id": numero_ley,
        "infoleg_id": infoleg_id,
        "source_url": source_url,
        "sha256_hash": sha256_hash,
    }

    units: list[dict] = []
    current_kind = "preambulo"
    current_title = "PREAMBULO"
    buffer: list[str] = []
    heading_stack: list[str] = []

    def flush() -> None:
        nonlocal buffer, current_kind, current_title
        text = " ".join(buffer).strip()
        if text:
            units.append(
                {
                    "kind": current_kind,
                    "title": current_title,
                    "path": " > ".join(heading_stack),
                    "text": text,
                    **base_meta,
                }
            )
        buffer = []

    for elem in soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "b"]):
        text = elem.get_text(separator=" ", strip=True)
        if not text or len(text) < 3:
            continue

        if article_re.match(text):
            flush()
            current_kind = "articulo"
            current_title = text[:160]
            buffer.append(text)
            continue

        m = heading_re.match(text)
        if m:
            flush()
            current_kind = m.group(1).lower()
            current_title = text[:160]
            heading_stack.append(current_title)
            buffer.append(text)
            continue

        buffer.append(text)

    flush()
    return units


def cargar_ley_infoleg(numero_ley: int) -> tuple[list[dict], dict]:
    """
    Carga una ley sancionada desde Infoleg siguiendo el flujo oficial:
    1) buscarNormas.do -> 2) verNorma.do?id=... -> 3) texact.htm
    """
    infoleg_id, search_url = _buscar_id_norma_infoleg(numero_ley)
    ver_norma_url = _url_ver_norma(infoleg_id)

    ver_resp = requests.get(ver_norma_url, timeout=20)
    ver_resp.raise_for_status()
    ver_resp.encoding = ver_resp.apparent_encoding or "utf-8"

    url_texact = _extraer_url_texact_desde_ver_norma(ver_resp.text, ver_norma_url)
    if not url_texact:
        url_texact = _url_texact_fallback(infoleg_id)

    texact_resp = requests.get(url_texact, timeout=30)
    texact_resp.raise_for_status()
    texact_resp.encoding = texact_resp.apparent_encoding or "utf-8"

    sha256_hash = hashlib.sha256(texact_resp.content).hexdigest()
    soup = BeautifulSoup(texact_resp.text, "html.parser")

    article_units = _parse_infoleg_html(
        soup,
        numero_ley=numero_ley,
        infoleg_id=infoleg_id,
        source_url=url_texact,
        sha256_hash=sha256_hash,
    )

    title_text = soup.title.get_text(strip=True) if soup.title else f"Ley {numero_ley}"

    metadata_doc = {
        "source": "INFOLEG",
        "status": "VIGENTE",
        "law_id": numero_ley,
        "infoleg_id": infoleg_id,
        "source_url": url_texact,
        "search_url": search_url,
        "ver_norma_url": ver_norma_url,
        "sha256_hash": sha256_hash,
        "titulo": title_text,
    }

    return article_units, metadata_doc


print("Módulo A (Infoleg) cargado OK con flujo buscarNormas -> verNorma -> texact.")

Módulo A (Infoleg) cargado OK con flujo buscarNormas -> verNorma -> texact.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# ROUTING — cambiar estos dos valores según el documento a procesar
# ─────────────────────────────────────────────────────────────────────────────
tipo_documento = "sancion"   # "sancion" | "proyecto"
numero_ley     = 20744       # solo se usa si tipo_documento == "sancion"

# ─────────────────────────────────────────────────────────────────────────────
if tipo_documento == "sancion":
    article_units, doc_metadata = cargar_ley_infoleg(numero_ley)

    print(f"Ley {numero_ley} cargada desde Infoleg.")
    print(f"Título:        {doc_metadata.get('titulo', '')}")
    print(f"Buscar URL:    {doc_metadata.get('search_url', '')}")
    print(f"Ver norma URL: {doc_metadata.get('ver_norma_url', '')}")
    print(f"Source URL:    {doc_metadata.get('source_url', '')}")
    print(f"SHA-256:       {doc_metadata.get('sha256_hash', '')}")
    print(f"Unidades:      {len(article_units)}")
    print(f"Artículos:     {sum(1 for u in article_units if u['kind'] == 'articulo')}")
    print("\n→ Saltear la Sección 8 (OCR pipeline). No es necesaria.")

else:
    print("Documento es Proyecto (BORA/OCR) → continuar con la Sección 8.")

Ley 20744 cargada desde Infoleg.
Título:        INFOLEG
Buscar URL:    https://servicios.infoleg.gob.ar/infolegInternet/buscarNormas.do?tipoNorma=1&numero=20744&anioSancion=
Ver norma URL: https://servicios.infoleg.gob.ar/infolegInternet/verNorma.do?id=25552
Source URL:    https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm
SHA-256:       1e051cf62df49b25c5f5796295e8bdccd738e65112939ee30425e3a2c33b89a8
Unidades:      666
Artículos:     592

→ Saltear la Sección 8 (OCR pipeline). No es necesaria.


### 8. Limpieza y normalización legal
Este bloque remueve ruido de ingesta, protege encabezados jurídicos reales y deja el texto en unidades legales inspeccionables antes de pasar a chunking final.

### 8a. Diccionario legal y scoring de confianza OCR
Antes de descartar una línea "ambigua", le asignamos un **score de confianza** (0.0 = basura, 1.0 = confiable).
Esto nos permite:
- Descartar líneas severamente dañadas (score < 0.6)
- Revisar manualmente dudosas (0.6 ≤ score < 0.8)
- Aceptar líneas limpias (score ≥ 0.8)


In [ ]:
import re

# Diccionario básico de español + términos legales argentinos
LEGAL_SPANISH_DICT = {
    # Frecuentes
    "el", "la", "de", "que", "y", "a", "en", "se", "del", "con", "por", "para",
    "un", "una", "es", "este", "ese", "cual", "cuyo", "mas", "más", "no",
    "los", "las", "o", "cuando", "si", "donde", "como", "cuales",

    # Órganos e instrumentos
    "constitucion", "constitución", "nacional", "honorable",
    "camara", "cámara", "senadores", "diputados", "senado", "congreso", "parlamento",

    # Jurídico
    "articulo", "artículo", "inciso", "párrafo", "parágrafo", "apartado",
    "proyecto", "ley", "decreto", "resolucion", "resolución", "disposicion", "disposición",
    "expediente", "expedinte", "expedieente",  # variaciones OCR
    "presentado", "presentada", "presentados",
    "cuyo", "acto", "fecha", "firma", "firmante", "numerado",
    "aprobado", "sancionado", "sanción", "vigencia", "deroga", "derogacion", "abrogación",
    "abroga", "abrogaciones", "disposiciones", "precedentes", "contrarias",
    "derecho", "derechos", "obligacion", "obligaciones", "obligatoriedad",
    "persona", "personas", "ciudadano", "ciudadana", "argentino", "argentina",
    "empresa", "entidad", "institucion", "institución", "organismo", "organismos",
    "ministro", "ministra", "secretario", "secretaria", "secretaría",
    "articulos",

    # Verbos y conectores
    "segun", "según", "mediante", "establece", "deber", "consideracion", "consideración",
    "teniendo", "resulta", "resultan", "visto", "considerando", "entiende", "entienden",
    "durante", "dentro", "fuera", "frente", "ante",

    # Ordinales y numeración
    "primero", "primera", "segundo", "segunda", "tercero", "tercera",
    "numero", "número", "numeración",
}


def _is_protected_legal_line(line: str) -> bool:
    """True si la línea parece contener contenido legal válido que no se debe descartar."""
    legal_keywords = [
        "artículo", "articulo", "ley", "decreto", "resolución", "resolucion", "proyecto",
        "honorable", "cámara", "camara", "senado", "diputados", "congreso",
        "constitución", "constitucion", "nacional", "sancionado", "vigencia",
    ]
    line_lower = line.lower()
    return any(keyword in line_lower for keyword in legal_keywords)


def _ocr_confidence(line: str, context_lines: list[str] = None, legal_dict: set = None) -> float:
    """Calcula confianza de que la línea es contenido legítimo (0.0=basura, 1.0=confiable)."""
    if legal_dict is None:
        legal_dict = LEGAL_SPANISH_DICT

    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return 1.0

    score = 1.0

    # Penalización 1: ruido de símbolos frecuentes
    noise_chars = len(re.findall(r"[~_|{}\[\]`¬]", l))
    score -= min(0.3, noise_chars * 0.15)

    # Penalización 2: densidad de caracteres inusuales
    unusual = len(re.findall(r"[~_\\|{}\[\]`¬@#$%&*]", l))
    if l:
        unusual_ratio = unusual / len(l)
        if unusual_ratio > 0.2:
            score -= 0.4

    # Penalización 3: palabras raras
    words = re.findall(r"\b[a-záéíóúñ]+\b", l, re.IGNORECASE)
    words_lower = [w.lower() for w in words]

    if words:
        valid_words = sum(1 for w in words_lower if w in legal_dict or len(w) > 3)
        valid_ratio = valid_words / len(words)
        if valid_ratio < 0.5:
            score -= (1 - valid_ratio) * 0.3

    # Penalización 4: cambios de caso (garbled OCR)
    if len(l) < 80 and re.search(r"[a-z][A-Z][a-z]", l):
        score -= 0.25

    # Bonificación 1: vocabulario legal
    if words_lower and any(w in legal_dict for w in words_lower if len(w) > 3):
        score += 0.15

    # Bonificación 2: aparición en contexto cercano
    if context_lines and words_lower:
        context_text = " ".join(context_lines).lower()
        if any(w in context_text for w in words_lower if len(w) > 4):
            score += 0.1

    return max(0.0, min(1.0, score))


# Prueba rápida del scorer
test_lines = [
    "Artículo 1. Objeto de la Ley",        # esperado alto
    "P99~ct cíe Pilra, o t ( 4 b",          # esperado bajo
    "El presente proyecto de ley regula",  # esperado alto
    "0E7-700/25",                          # esperado bajo
    "Honorable Cámara de Diputados",       # esperado alto
]

print("=== Test OCR Confidence ===")
for line in test_lines:
    conf = _ocr_confidence(line, legal_dict=LEGAL_SPANISH_DICT)
    print(f"Score {conf:.2f}: {line[:50]}")


In [ ]:
import re
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# Fase 0 — Extracción de firmas digitales (llevan a metadatos, no al cuerpo)
# ─────────────────────────────────────────────────────────────────────────────
_SIGNATURE_RE = re.compile(
    r"Digitally\s+signed\s+by\b[^\n]+"
    r"(?:\n(?![ \t]*\n)(?!(?:Art[íi]culo|T[íi]tulo|Cap[íi]tulo)\b)[^\n]{1,120}){0,7}",
    re.IGNORECASE,
)


def extract_signature_blocks(text: str) -> tuple[str, list[str]]:
    signatures: list[str] = []

    def _collect(m: re.Match) -> str:
        signatures.append(m.group(0).strip())
        return ""

    cleaned = _SIGNATURE_RE.sub(_collect, text)
    return re.sub(r"\n{3,}", "\n\n", cleaned).strip(), signatures


# ─────────────────────────────────────────────────────────────────────────────
# Fase 0b — Unión de palabras cortadas por salto de línea
# ─────────────────────────────────────────────────────────────────────────────
def _join_hyphenated_linebreaks(text: str) -> str:
    return re.sub(
        r"([A-Za-záéíóúñÁÉÍÓÚÑ])-\n[ \t]*([A-Za-záéíóúñÁÉÍÓÚÑ])",
        r"\1\2",
        text,
    )


def _normalize_confusable_caps(line: str) -> str:
    """Corrige OCR en tokens casi-mayúscula con dígitos (V1LLARRUEL -> VILLARRUEL)."""
    mapping = str.maketrans({"0": "O", "1": "I", "5": "S"})

    def _fix_token(m: re.Match) -> str:
        token = m.group(0)
        return token.translate(mapping)

    return re.sub(r"\b[\wÁÉÍÓÚÑ]{4,}\b", lambda m: _fix_token(m) if any(ch.isdigit() for ch in m.group(0)) else m.group(0), line)


# ─────────────────────────────────────────────────────────────────────────────
# Helpers de detección y normalización línea a línea
# ─────────────────────────────────────────────────────────────────────────────
def _norm_line(line: str) -> str:
    return re.sub(r"\s+", " ", line).strip().lower()


def _extract_candidate_header_footer_lines(page_text: str, top_n: int = 4, bottom_n: int = 4) -> list[str]:
    lines = [ln.strip() for ln in page_text.splitlines() if ln.strip()]
    if not lines:
        return []
    head = lines[:top_n]
    tail = lines[-bottom_n:] if len(lines) > top_n else []
    return head + tail


def _is_protected_legal_line(line: str) -> bool:
    l = line.strip()
    if not l:
        return False
    if re.match(r"^(art[íi]culo|art\.)\s*\d+", l, re.IGNORECASE):
        return True
    if re.match(r"^(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", l, re.IGNORECASE):
        return True
    return False


def detect_boilerplate_lines_from_pages(docs: list, min_repeat_pages: int = 2) -> set[str]:
    counter: Counter[str] = Counter()
    for d in docs:
        seen_this_page: set[str] = set()
        for ln in _extract_candidate_header_footer_lines(d.page_content):
            if _is_protected_legal_line(ln):
                continue
            n = _norm_line(ln)
            if not n or len(n) < 6:
                continue
            seen_this_page.add(n)
        for n in seen_this_page:
            counter[n] += 1
    return {ln for ln, freq in counter.items() if freq >= min_repeat_pages}


def is_obvious_header_footer_line(line: str) -> bool:
    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return False
    # Expedientes reales y variantes OCR (OD, 0D, 0E7, etc.)
    if re.search(r"\b(?:[A-Z0-9]{1,4})-\d{2,5}/\d{2,4}\b", l, re.IGNORECASE):
        return True
    if len(l) <= 28 and len(re.findall(r"[^\w\s]", l)) >= 4:
        return True
    if re.search(r"aniversario|palacio legislativo|h\.?\s*camara|c[aá]mara|gde", l, re.IGNORECASE):
        return True
    return False


def _is_obvious_stamp_line(line: str) -> bool:
    """Detecta líneas de sello/margen/paginación típicas del OCR roto."""
    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return False
    if re.fullmatch(r"[•·.\-_*~]+", l):
        return True
    if re.fullmatch(r"\d{1,3}", l):
        return True
    # Patrón tipo _4 4- o similares de margen
    if re.fullmatch(r"_?\d+(?:\s+\d+)?-?", l):
        return True
    # Mucho símbolo + dígitos en línea corta
    sym = len(re.findall(r"[~_\\|{}\[\]<>@#$%^&*.,;:()\-]", l))
    dig = len(re.findall(r"\d", l))
    if len(l) < 70 and sym >= 4 and dig >= 2:
        return True
    # Garbled alfanumérico con slashes y puntuación
    if len(l) < 80 and re.search(r"[A-Za-z].*[~/\\].*\d|\d.*[~/\\].*[A-Za-z]", l):
        return True
    # Mixed case + slash/coma/guion (ej: cediAde/i-bef,a)
    if len(l) < 80 and re.search(r"[a-z][A-Z]|[A-Z][a-z][A-Z]", l) and re.search(r"[/,\-_]", l):
        return True
    return False


def _is_ocr_garbage_line(line: str, confidence_threshold: float = 0.6, legal_dict: set = None) -> bool:
    if legal_dict is None:
        legal_dict = LEGAL_SPANISH_DICT
    confidence = _ocr_confidence(line, context_lines=None, legal_dict=legal_dict)
    return confidence < confidence_threshold


def normalize_ocr_line(line: str) -> str:
    # 1) Unicode/control
    line = line.replace("\ufffd", "")
    line = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x80-\x9f]", "", line)
    # 2) escapes OCR
    line = line.replace("\u00ad", "")
    line = line.replace("`", "")
    line = re.sub(r"([A-Za-zÁÉÍÓÚáéíóúñÑ])\\'([A-Za-zÁÉÍÓÚáéíóúñÑ])", r"\1\2", line)  # co\'n -> con
    line = re.sub(r"\\(['\"])", r"\1", line)
    # 3) ordinales
    line = re.sub(r"(?<=\d)\s*[º°o]\b", "°", line)
    line = re.sub(r"(?i)\b(art[íi]culo\s+)([1-9])0\b", r"\1\2°", line)  # Artículo 80 -> 8°
    # 4) basura de puntuación suelta
    line = re.sub(r"([A-Za-záéíóúñ])\)\.", r"\1.", line)  # come). -> come.
    # 5) confusiones alfanuméricas en mayúsculas
    line = _normalize_confusable_caps(line)
    # 6) espacios
    line = re.sub(r"\s+", " ", line).strip()
    return line


def _reflow_preserving_legal_structure(lines: list[str]) -> str:
    out: list[str] = []
    buf: list[str] = []

    def flush_buf() -> None:
        nonlocal buf
        if buf:
            out.append(" ".join(buf).strip())
            buf = []

    for line in lines:
        if not line:
            flush_buf()
            if out and out[-1] != "":
                out.append("")
            continue
        if _is_protected_legal_line(line):
            flush_buf()
            out.append(line)
            continue
        buf.append(line)

    flush_buf()
    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _dedup_near_duplicate_paragraphs(text: str) -> str:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    kept: list[str] = []
    seen_norms: list[str] = []

    def norm(p: str) -> str:
        p = p.lower()
        p = re.sub(r"\s+", " ", p)
        p = re.sub(r"[^\w\sáéíóúñ]", "", p)
        return p.strip()

    for p in paragraphs:
        n = norm(p)
        if not n:
            continue
        if n in seen_norms:
            continue
        overlapped = False
        for old_n in seen_norms:
            if len(n) > 180 and len(old_n) > 180 and (n in old_n or old_n in n):
                overlapped = True
                break
        if overlapped:
            continue
        kept.append(p)
        seen_norms.append(n)

    return "\n\n".join(kept)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline principal de limpieza
# ─────────────────────────────────────────────────────────────────────────────
def clean_pdf_text_remove_headers_footers(docs: list, confidence_threshold: float = 0.6) -> tuple[str, list[str], dict]:
    boilerplate = detect_boilerplate_lines_from_pages(docs, min_repeat_pages=max(2, len(docs) // 4))
    cleaned_pages: list[str] = []
    all_signatures: list[str] = []
    seal_lines: list[str] = []
    stats = {
        "lines_processed": 0,
        "lines_boilerplate": 0,
        "lines_header_footer": 0,
        "lines_stamps": 0,
        "lines_ocr_garbage": 0,
        "lines_kept": 0,
    }

    for d in docs:
        page_text = _join_hyphenated_linebreaks(d.page_content)
        page_text, page_sigs = extract_signature_blocks(page_text)
        all_signatures.extend(page_sigs)

        kept_lines: list[str] = []
        for raw in page_text.splitlines():
            line = raw.strip()
            stats["lines_processed"] += 1

            if not line:
                kept_lines.append("")
                continue

            if _is_protected_legal_line(line):
                kept_lines.append(normalize_ocr_line(line))
                stats["lines_kept"] += 1
                continue

            n = _norm_line(line)
            if n in boilerplate:
                stats["lines_boilerplate"] += 1
                seal_lines.append(line)
                continue
            if is_obvious_header_footer_line(line):
                stats["lines_header_footer"] += 1
                seal_lines.append(line)
                continue
            if _is_obvious_stamp_line(line):
                stats["lines_stamps"] += 1
                seal_lines.append(line)
                continue
            if _is_ocr_garbage_line(line, confidence_threshold=confidence_threshold, legal_dict=LEGAL_SPANISH_DICT):
                stats["lines_ocr_garbage"] += 1
                seal_lines.append(line)
                continue

            kept_lines.append(normalize_ocr_line(line))
            stats["lines_kept"] += 1

        page_clean = _reflow_preserving_legal_structure(kept_lines)
        cleaned_pages.append(page_clean)

    cleaned_text = "\n\n".join(cleaned_pages)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text).strip()
    cleaned_text = _dedup_near_duplicate_paragraphs(cleaned_text)

    # Guardrail final: limpia islas de ruido entre párrafos legales
    cleaned_text = re.sub(
        r"\n\n(?:[\W\d_~\\/.,;:()\-]{2,}|[A-Za-z0-9~_/.,;:()\-]{1,40})\n\n(?=(?:Art[íi]culo|T[íi]tulo|Cap[íi]tulo)\b)",
        "\n\n",
        cleaned_text,
        flags=re.IGNORECASE,
    )

    stats["seal_lines_detected"] = len(seal_lines)
    stats["seal_lines_sample"] = seal_lines[:10]
    return cleaned_text, all_signatures, stats


# ─────────────────────────────────────────────────────────────────────────────
# Segmentación en unidades legales
# ─────────────────────────────────────────────────────────────────────────────
def split_legal_articles(text: str) -> list[dict]:
    lines = text.splitlines()
    sections: list[dict] = []
    current_kind = "preambulo"
    current_title = "PREAMBULO"
    buffer: list[str] = []

    article_re = re.compile(r"^\s*(art[íi]culo|art\.)\s*([\dA-Za-zº°./-]+)", re.IGNORECASE)
    heading_re = re.compile(r"^\s*(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", re.IGNORECASE)
    heading_stack: list[str] = []

    def flush():
        nonlocal buffer, current_kind, current_title
        content = "\n".join(buffer).strip()
        if content:
            sections.append({
                "kind": current_kind,
                "title": current_title,
                "path": " > ".join(heading_stack),
                "text": content,
            })
        buffer = []

    for raw_line in lines:
        line = normalize_ocr_line(raw_line)
        if not line:
            buffer.append("")
            continue

        article_match = article_re.match(line)
        if article_match:
            flush()
            current_kind = "articulo"
            current_title = line[:160]
            buffer.append(line)
            continue

        heading_match = heading_re.match(line)
        if heading_match:
            flush()
            current_kind = heading_match.group(1).lower()
            current_title = line[:160]
            heading_stack.append(current_title)
            buffer.append(line)
            continue

        buffer.append(line)

    flush()
    return sections


# ─────────────────────────────────────────────────────────────────────────────
# Ejecución del pipeline
# ─────────────────────────────────────────────────────────────────────────────
pdf_text_clean, doc_signatures, cleaning_stats = clean_pdf_text_remove_headers_footers(
    docs,
    confidence_threshold=0.65,
)
article_units = split_legal_articles(pdf_text_clean)

print("Chars original:", len(pdf_text))
print("Chars limpio:", len(pdf_text_clean))
print("Firmas extraídas:", len(doc_signatures))
print("Sello/ruido extraído:", cleaning_stats.get("seal_lines_detected", 0))
print("Unidades legales detectadas:", len(article_units))
print("Articulos detectados:", sum(1 for item in article_units if item["kind"] == "articulo"))
print("\n=== Estadísticas de limpieza ===")
for key, val in cleaning_stats.items():
    if key != "seal_lines_sample":
        print(f"  {key}: {val}")

print("\n=== Muestra de sello/ruido extraído ===")
for s in cleaning_stats.get("seal_lines_sample", [])[:5]:
    print(" -", s)


### 9. Inspección del resultado normalizado
Estas celdas muestran el texto limpio y las unidades legales detectadas para revisar manualmente si la normalización conservó el contenido jurídico correcto.

In [ ]:
# Vista rápida del texto limpio y normalizado
pdf_text_clean


In [ ]:
len(article_units)

# Muestra de unidades legales normalizadas
for i, item in enumerate(article_units[:5], start=1):
    print(f"#{i} | {item['kind']} | {item['title']}")
    if item["path"]:
        print("path:", item["path"])
    print(item["text"][:500])
    print("-" * 80)

### 10. Payload de ingesta al backend
Acá se arma el payload que después se puede mandar al endpoint de ingesta para comparar el preprocesamiento del notebook contra el pipeline del backend.

In [ ]:
noise_samples = [
    "cediAde/i-bef,a",
    "P9 9~",
    "P99~ct cíe Pilra",
    "0E7-700/25",
    "0D-700/25",
    "_4 4-",
    "V1LLARRUEL",
    "co\\'n",
    "come).",
]

print("=== Chequeo de patrones de ruido ===")
for s in noise_samples:
    found = s in pdf_text_clean
    print(f"{s!r}: {'ENCONTRADO' if found else 'OK (no aparece)'}")

print("\n=== Chequeos esperados de normalización ===")
print("'VILLARRUEL' presente:", "VILLARRUEL" in pdf_text_clean)
print("'V1LLARRUEL' presente:", "V1LLARRUEL" in pdf_text_clean)
print("'Artículo 8°' presente:", "Artículo 8°" in pdf_text_clean or "ARTÍCULO 8°" in pdf_text_clean)
print("'Artículo 80' presente:", "Artículo 80" in pdf_text_clean or "ARTÍCULO 80" in pdf_text_clean)

print("\n=== Sello/Firma a metadatos ===")
print("Firmas extraídas:", len(doc_signatures))
print("Líneas sello/ruido extraídas:", cleaning_stats.get("seal_lines_detected", 0))


### 9a. Chequeo puntual de ruido reportado
Validación rápida para confirmar si los patrones concretos detectados en ingesta siguen presentes tras la limpieza.

In [ ]:
# 2) Ingesta de documento desde path local (PDF/imagen/texto)
local_pdf_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf"  # <- cambia esto

ingest_payload = {
    'file_path': local_pdf_path,
    # 'minio_path': 'minio://my-bucket/contexto/ley-ejemplo.pdf',  # alternativa
    'presentado_por': 'Dip. Juan Perez',
    'proyecto_tipo': 'modificacion',  # 'base' o 'modificacion'
    'ley_base': 'Ley 27.275',
    'iniciado_en': 'Camara de Diputados',
    'expediente_diputados': '1234-D-2026',
    'expediente_senado': '5678-S-2026',
    'publicado_en': 'Boletin Oficial de la Republica Argentina',
    'fecha': '2026-04-10',
    'ley_numero': '27.804',
    'chunk_size': 700,
    'chunk_overlap': 120,
    'replace_existing': True,
    'metadata': {'fuente': 'manual-notebook', 'tipo': 'normativa'}
}


## Capa de noticias (complementaria a leyes por n8n + MQ)

Este bloque agrega una fuente de noticias para enriquecer el contexto.

- La fuente principal de verdad siguen siendo las leyes que llegan por n8n/MQ.
- Las noticias se indexan por separado y se usan como contexto adicional.
- Luego calculamos un indice de verdad comparando afirmaciones contra el corpus legal.

### 11. Dependencias para capa de noticias
Este primer paso de la capa complementaria instala el parser de RSS que vamos a usar para traer artículos periodísticos.

In [ ]:
%pip install -q feedparser

### 12. Búsqueda de URLs de noticias
Estas celdas consultan Google News RSS y generan una lista deduplicada de links por tema para enriquecer el contexto con prensa.

In [ ]:
import feedparser
from urllib.parse import quote

# Google News RSS por tema (sin API key)
temas = [
    "congreso argentina ley inteligencia artificial",
    "congreso argentina regulacion datos personales",
    "camara diputados tecnologia argentina",
]


def buscar_urls_noticias(temas, max_por_tema=5):
    urls = []
    for tema in temas:
        rss_url = f"https://news.google.com/rss/search?q={quote(tema)}&hl=es-419&gl=AR&ceid=AR:es-419"
        feed = feedparser.parse(rss_url)
        for entry in feed.entries[:max_por_tema]:
            if hasattr(entry, "link"):
                urls.append(entry.link)

    # Deduplicar manteniendo orden
    return list(dict.fromkeys(urls))


news_urls = buscar_urls_noticias(temas, max_por_tema=5)
len(news_urls), news_urls[:3]

### 13. Descarga de contenidos periodísticos
Con la lista de URLs armada, este bloque usa un loader web para bajar el contenido bruto de las noticias seleccionadas.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader_noticias = WebBaseLoader(web_path=news_urls)
docs_noticias_raw = loader_noticias.load()

len(docs_noticias_raw)

### 14. Limpieza del texto de noticias
Acá se remueve ruido típico de sitios periodísticos para quedarnos con texto utilizable antes de crear embeddings.

In [ ]:
from langchain_core.documents import Document
import re


# Limpieza simple para reducir ruido de navegación/publicidad
def limpiar_texto_noticia(texto: str) -> str:
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(
        r"(Leia também|Publicidade|Assine|Compartilhe).*",
        "",
        texto,
        flags=re.IGNORECASE,
    )
    return texto.strip()


docs_noticias = []
for d in docs_noticias_raw:
    texto_limpo = limpiar_texto_noticia(d.page_content)
    if len(texto_limpo) < 400:
        continue

    meta = dict(d.metadata) if d.metadata else {}
    meta["tipo_fonte"] = "noticia"
    docs_noticias.append(Document(page_content=texto_limpo, metadata=meta))

len(docs_noticias)

### 15. Chunking de noticias
Este paso divide las noticias limpias en fragmentos manejables para indexarlas y recuperarlas como contexto complementario.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

news_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
)

news_chunks = news_splitter.split_documents(docs_noticias)
len(news_chunks)

### 16. Vector store y retrieval de noticias
Aquí se construye el índice vectorial de noticias y un retriever simple para probar búsquedas temáticas.

In [ ]:
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

# Embeddings con OpenAI (evita depender de HFEmbeddingsAdapter no definido)
embeddings = OpenAIEmbeddings(
    api_key=openai_api_key,
    model="text-embedding-3-small",
)

vectorstore_noticias = InMemoryVectorStore.from_documents(
    documents=news_chunks,
    embedding=embeddings,
)

retriever_noticias = vectorstore_noticias.as_retriever(search_kwargs={"k": 3})
retriever_noticias.invoke("ley de inteligencia artificial argentina")[:1]

### 17. Evaluación de índice de verdad
Este bloque define el prompt y la función que comparan afirmaciones periodísticas contra el contexto legal recuperado.

In [ ]:
import json
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt_indice_verdad = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Eres un verificador de consistencia.
Debes puntuar de 0 a 1 cuanto coincide una afirmacion periodistica con el contexto legal.
Responde SOLO JSON valido con este formato:
{"indice_verdad": 0.0, "justificacion": "texto breve"}
""",
        ),
        (
            "human",
            "Afirmacion: {afirmacion}\n\nContexto legal:\n{contexto_leyes}",
        ),
    ]
)

cadena_indice_verdad = prompt_indice_verdad | modelo | StrOutputParser()


def calcular_indice_verdad(afirmacion: str, k_leyes: int = 3):
    leyes_rel = retriever.invoke(afirmacion)[:k_leyes]
    contexto_leyes = "\n\n".join([d.page_content for d in leyes_rel])

    raw = cadena_indice_verdad.invoke(
        {"afirmacion": afirmacion, "contexto_leyes": contexto_leyes}
    )

    try:
        resultado = json.loads(raw)
    except Exception:
        resultado = {"indice_verdad": None, "justificacion": raw}

    resultado["afirmacion"] = afirmacion
    return resultado


# Ejemplo de uso
calcular_indice_verdad("El Congreso ya aprobo una ley integral de IA en Argentina")